In [1]:
!pip install -q ultralytics deep-sort-realtime opencv-python-headless gradio rarfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 91.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

PROJECT_PATH = "/content/drive/MyDrive/WeaponDetectionProject"
WEIGHTS_DIR  = os.path.join(PROJECT_PATH, "weapon_model", "weights")
BEST_WEIGHTS = os.path.join(WEIGHTS_DIR, "best.pt")
LAST_WEIGHTS = os.path.join(WEIGHTS_DIR, "last.pt")
os.makedirs(PROJECT_PATH, exist_ok=True)

print(f"Project folder : {PROJECT_PATH}")
print(f"Best weights    : {BEST_WEIGHTS}")
print(f"Last checkpoint : {LAST_WEIGHTS}")

if os.path.exists(BEST_WEIGHTS):
    print("Trained model FOUND on Drive.")
else:
    print("No trained model found.")

Mounted at /content/drive
Project folder : /content/drive/MyDrive/WeaponDetectionProject
Best weights    : /content/drive/MyDrive/WeaponDetectionProject/weapon_model/weights/best.pt
Last checkpoint : /content/drive/MyDrive/WeaponDetectionProject/weapon_model/weights/last.pt
Trained model FOUND on Drive.


In [4]:
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DATASET_ZIP = '/content/Gun_data_labeled.zip'
EXTRACT_DIR  = Path('/content/dataset')
YAML_PATH    = '/content/weapon_data.yaml'

def find_dataset_root(base: Path):
    """Find directory containing both 'images' and 'labels'."""
    for p in sorted(base.rglob('images')):
        if (p.parent / 'labels').exists():
            return p.parent
    return None

if not os.path.exists(DATASET_ZIP):
    raise FileNotFoundError(f"File not found at {DATASET_ZIP}")

!rm -rf /content/dataset
!mkdir -p /content/dataset

print("Extracting dataset...")
!unzip -q "$DATASET_ZIP" -d /content/dataset

print("Extraction attempt complete")
root = find_dataset_root(EXTRACT_DIR)

n_images = len(list((root / 'images').rglob('*.jpg'))) + \
           len(list((root / 'images').rglob('*.png')))
n_labels = len(list((root / 'labels').rglob('*.txt')))

print(f"   Images: {n_images} | Labels: {n_labels}")

yaml_content = f"""path: {root}
train: images
val: images

nc: 1
names:
  0: weapon
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)
print(f"YAML written to {YAML_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting dataset...
Extraction attempt complete
   Images: 3000 | Labels: 2
YAML written to /content/weapon_data.yaml


In [ ]:

import os, shutil
from pathlib import Path
from ultralytics import YOLO

DATASET_DIR = Path('/content/dataset')
YAML_PATH   = '/content/weapon_data.yaml'
PROJECT_PATH = "/content/drive/MyDrive/WeaponDetectionProject"
LAST_WEIGHTS = os.path.join(PROJECT_PATH, "weapon_model", "weights", "last.pt")

shutil.rmtree(DATASET_DIR / "__MACOSX", ignore_errors=True)

def find_dataset_root(base: Path):
    for p in sorted(base.rglob('images')):
        if "__MACOSX" in str(p):
            continue
        if (p.parent / 'labels').exists():
            return p.parent
    return None

root = find_dataset_root(DATASET_DIR)

yaml_content = f"""path: {root}
train: images
val: images

nc: 1
names:
  0: weapon
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

import torch
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

if os.path.exists(LAST_WEIGHTS):
    print(f"🔄 Resuming training from {LAST_WEIGHTS}")
    model = YOLO(LAST_WEIGHTS)
    model.train(
        resume=True,
        device=DEVICE,
        project=PROJECT_PATH,
        name="weapon_model",
        exist_ok=True
    )
else:
    print("🚀 Starting fresh training...")
    model = YOLO('yolov8m.pt')
    model.train(
        data=YAML_PATH,
        epochs=50,
        imgsz=640,
        batch=16,
        device=DEVICE,
        project=PROJECT_PATH,
        name="weapon_model",
        exist_ok=True,
        save_period=5
    )

In [5]:

import os
from ultralytics import YOLO, RTDETR

PROJECT_PATH = "/content/drive/MyDrive/WeaponDetectionProject"
BEST_WEIGHTS = os.path.join(PROJECT_PATH, "weapon_model", "weights", "best.pt")

if not os.path.exists(BEST_WEIGHTS):
    raise FileNotFoundError(
        f" No trained model found at {BEST_WEIGHTS}.\n"
        "Please run Cells 3 and 4 to train the model first."
    )

WEAPON_MODEL = YOLO(BEST_WEIGHTS)
print(f" Weapon model loaded from: {BEST_WEIGHTS}")

GENERAL_MODEL = RTDETR("rtdetr-l.pt")
print("RT-DETR general model loaded")

COCO_NAMES = GENERAL_MODEL.names
print(f"   COCO classes available: {len(COCO_NAMES)}")

 Weapon model loaded from: /content/drive/MyDrive/WeaponDetectionProject/weapon_model/weights/best.pt
RT-DETR general model loaded
   COCO classes available: 80


In [6]:

import shutil
from pathlib import Path

DATASET_DIR = Path('/content/dataset')
YAML_PATH   = '/content/weapon_data.yaml'

shutil.rmtree(DATASET_DIR / "__MACOSX", ignore_errors=True)

for cache in DATASET_DIR.rglob("*.cache"):
    cache.unlink()

def find_dataset_root(base: Path):
    for p in sorted(base.rglob('images')):
        if "__MACOSX" in str(p):
            continue
        if (p.parent / 'labels').exists():
            return p.parent
    return None

root = find_dataset_root(DATASET_DIR)

if root is None:
    raise RuntimeError("Dataset still broken")

print(f"Correct dataset root: {root}")

yaml_content = f"""path: {root}
train: images
val: images

nc: 1
names:
  0: weapona
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print("YAML FIXED (no __MACOSX)")

print("\n🔍 Checking paths:")
print("Images exist:", (root / "images").exists())
print("Labels exist:", (root / "labels").exists())

Correct dataset root: /content/dataset/data
YAML FIXED (no __MACOSX)

🔍 Checking paths:
Images exist: True
Labels exist: True


In [7]:

import numpy as np
import os

YAML_PATH    = '/content/weapon_data.yaml'
PROJECT_PATH = "/content/drive/MyDrive/WeaponDetectionProject"
BEST_WEIGHTS = os.path.join(PROJECT_PATH, "weapon_model", "weights", "best.pt")

try:
    WEAPON_MODEL
except NameError:
    from ultralytics import YOLO
    WEAPON_MODEL = YOLO(BEST_WEIGHTS)

print("Running validation to find optimal confidence threshold...")
metrics = WEAPON_MODEL.val(data=YAML_PATH, verbose=False)

p_curve  = metrics.box.p_curve
r_curve  = metrics.box.r_curve
conf_arr = metrics.box.curves_results[0] if hasattr(metrics.box, 'curves_results') else np.linspace(0, 1, p_curve.shape[-1])

p_mean = p_curve.mean(0)
r_mean = r_curve.mean(0)
f1_arr = 2 * p_mean * r_mean / (p_mean + r_mean + 1e-9)

best_idx       = int(np.argmax(f1_arr))
WEAPON_CONF    = float(np.linspace(0, 1, len(f1_arr))[best_idx])
WEAPON_CONF    = max(0.30, min(0.85, WEAPON_CONF))  # clamp to sane range
GENERAL_CONF   = max(0.45, WEAPON_CONF - 0.05)      # general model slightly lower

print(f"\n Auto-selected thresholds:")
print(f"   Weapon  model confidence : {WEAPON_CONF:.2f} ({WEAPON_CONF*100:.0f}%)")
print(f"   General model confidence : {GENERAL_CONF:.2f} ({GENERAL_CONF*100:.0f}%)")
print(f"   F1 at best threshold     : {f1_arr[best_idx]:.3f}")

Running validation to find optimal confidence threshold...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,886,080 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.1 ms, read: 814.2±528.0 MB/s, size: 716.2 KB)
val: Scanning /content/dataset/data/labels... 3000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3000/3000 842.4it/s 3.6s
val: /content/dataset/data/images/armas (1125).jpg: corrupt JPEG restored and saved
val: /content/dataset/data/images/armas (1217).jpg: corrupt JPEG restored and saved
val: /content/dataset/data/images/armas (1221).jpg: corrupt JPEG restored and saved
val: /content/dataset/data/images/armas (1224).jpg: corrupt JPEG restored and saved
val: /content/dataset/data/images/armas (1717).jpg: corrupt JPEG restored and saved
val: /content/dataset/data/images/armas (1759).jpg: corrupt JPEG restored and saved
val: /content/dataset/data/images/armas (2140).jpg: corrupt 

In [9]:
import cv2
import numpy as np
import tempfile
import gradio as gr
import os
from collections import defaultdict
from ultralytics import YOLO
# CONSTANTS
SPEED_LIMIT_KMH   = 90
SPEED_SMOOTH_N    = 7
LINE_FRAC         = 0.50

# counting BAND instead of a 1-pixel line
# Objects are counted when their centroid passes through this band (pixels wide)
# Wide enough to survive 1–2 frame detection gaps and fast-moving objects
LINE_BAND_HALF    = 30     # band = line_y ± 30px  (total 60 px wide)

# ghost memory: remember last known position of a track for N frames
# so a 1–2 frame detection gap doesn't lose the crossing state
GHOST_FRAMES      = 8      # frames to remember a vanished track's last side

# unconfirmed track buffer: hold detections without an ID and try to
# match them to the counting band so fast crossings aren't missed
UNCONFIRMED_BAND_MARGIN = LINE_BAND_HALF + 20   # px either side of line

BASE_PX_PER_METER = 8.0
ZONE_X_START_FRAC = 0.40
ZONE_X_END_FRAC   = 0.60
LOITER_FRAMES     = 90
LOITER_DIST_PX    = 120
WEAPON_ASSIGN_DIST= 180
# ================= SHOPLIFTING DETECTION UPGRADE =================
SHELF_X1_FRAC = 0.10
SHELF_X2_FRAC = 0.90
SHELF_Y1_FRAC = 0.20
SHELF_Y2_FRAC = 0.80

ITEM_DISAPPEAR_FRAMES = 15
PICKUP_DIST_PX = 120

# MODELS
PROJECT_PATH = "/content/drive/MyDrive/WeaponDetectionProject"
BEST_WEIGHTS = os.path.join(PROJECT_PATH, "weapon_model", "weights", "best.pt")

try:
    WEAPON_MODEL
    GENERAL_MODEL
except NameError:
    print("Loading models…")
    WEAPON_MODEL  = YOLO(BEST_WEIGHTS)
    GENERAL_MODEL = RTDETR("rtdetr-l.pt")

GENERAL_CONF = globals().get("GENERAL_CONF", 0.60)
WEAPON_CONF  = globals().get("WEAPON_CONF",  0.15)

SECURITY_CLASSES = [0, 1, 2, 3, 5, 7, 24, 26, 28]
TRAFFIC_CLASSES  = [1, 2, 3, 5, 7]
COCO_NAMES       = GENERAL_MODEL.names

COLOR_MAP = {
    'person':    (0,   255,   0),
    'car':       (255, 200,   0),
    'motorbike': (0,   200, 255),
    'bus':       (255, 100,   0),
    'truck':     (200,   0, 255),
    'bicycle':   (0,   255, 200),
    'bag':       (255, 255,   0),
    'weapon':    (0,     0, 255),
}

# HELPERS
def get_label(cls):
    name = COCO_NAMES.get(cls, "object").lower()
    if cls in [24, 26, 28]:
        return "bag"
    return name


def box_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def perspective_px_per_meter(cy: float, frame_height: int) -> float:
    scale = 1.0 + (cy / frame_height)   # 1.0 (top/far) … 2.0 (bottom/close)
    return BASE_PX_PER_METER / scale


def iou(b1, b2):
    xi1, yi1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    xi2, yi2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-6)


def fuse_detections(dets, iou_thresh=0.45):
    filtered = []
    for det in sorted(dets, key=lambda x: x[1], reverse=True):
        if all(iou(det[0], fd[0]) < iou_thresh for fd in filtered):
            filtered.append(det)
    return filtered


def draw_box(frame, box, label, conf, color, thickness=2):
    x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    tag = f"{label} {conf:.0%}"
    (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    label_y1 = max(y1 - th - 6, 0)
    label_y2 = max(y1, th + 6)
    cv2.rectangle(frame, (x1, label_y1), (x1 + tw + 4, label_y2), color, -1)
    cv2.putText(frame, tag, (x1 + 2, label_y2 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)


def draw_box_with_speed(frame, box, label, conf, speed_kmh, color):
    x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    tag = (f"{label} {conf:.0%}  {speed_kmh:.1f}km/h"
           if speed_kmh > 0 else f"{label} {conf:.0%}")
    (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 2)
    label_y1 = max(y1 - th - 6, 0)
    label_y2 = max(y1, th + 6)
    cv2.rectangle(frame, (x1, label_y1), (x1 + tw + 4, label_y2), color, -1)
    cv2.putText(frame, tag, (x1 + 2, label_y2 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 0, 0), 2)


def overlay_alerts(frame, alert_lines, footer=None):
    y = 30
    for line in alert_lines:
        cv2.putText(frame, line, (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)
        y += 28
    if footer:
        cv2.putText(frame, footer, (10, frame.shape[0] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)


def draw_counts(frame, counts, w, h):
    y_off = h - 20 - len(counts) * 22
    for lbl, cnt in sorted(counts.items()):
        cv2.putText(frame, f"{lbl}: {cnt}", (w - 170, y_off),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    COLOR_MAP.get(lbl, (255, 255, 255)), 2)
        y_off += 22


# ROBUST LINE-CROSSING COUNTER
class RobustLineCrossingCounter:
    def __init__(self, line_coord: float, axis: str = "y"):
        self.line_coord = line_coord
        self.axis       = axis
        self.band_lo    = line_coord - LINE_BAND_HALF
        self.band_hi    = line_coord + LINE_BAND_HALF

        self.prev_side   = {}
        self.counted_ids = set()
        self.labels      = {}

        self.ghosts = {}
        self.unconfirmed_boxes = []

        self._extra_counts = defaultdict(int)

    def _coord(self, centre):
        return centre[1] if self.axis == "y" else centre[0]

    def _side(self, centre):
        return 1 if self._coord(centre) >= self.line_coord else -1

    def _in_band(self, centre):
        c = self._coord(centre)
        return self.band_lo <= c <= self.band_hi

    def tick_ghosts(self):
        expired = [tid for tid, (_, _, ttl) in self.ghosts.items() if ttl <= 1]
        for tid in expired:
            del self.ghosts[tid]

        for tid in list(self.ghosts):
            side, c, ttl = self.ghosts[tid]
            self.ghosts[tid] = (side, c, ttl - 1)

    def update(self, tid, centre, label):
        side = self._side(centre)

        # Remove ghost if track is back
        if tid in self.ghosts:
            self.prev_side[tid] = self.ghosts[tid][0]
            del self.ghosts[tid]

        # Already counted → ignore
        if tid in self.counted_ids:
            self.prev_side[tid] = side
            return False

        # First time seeing this ID
        if tid not in self.prev_side:
            # Try ghost match
            for gid, (gside, gc, _) in self.ghosts.items():
                if np.hypot(centre[0]-gc[0], centre[1]-gc[1]) < 80:
                    self.prev_side[tid] = gside
                    del self.ghosts[gid]
                    break

            if tid not in self.prev_side:
                self.prev_side[tid] = side
                return False

        prev = self.prev_side[tid]
        crossed = (prev != side)

        if crossed:
            self.counted_ids.add(tid)
            self.labels[tid] = label
            self.prev_side[tid] = side
            return True

        self.prev_side[tid] = side
        return False

    def expire_track(self, tid, centre):
        if tid not in self.counted_ids and tid in self.prev_side:
            self.ghosts[tid] = (self.prev_side[tid], centre, GHOST_FRAMES)

    def update_unconfirmed(self, centre, label, box):
        for prev_box in self.unconfirmed_boxes:
            if iou(prev_box, box) > 0.5:
                return False  # same object

        if self._in_band(centre):
            self.unconfirmed_boxes.append(box)
            if len(self.unconfirmed_boxes) > 50:
                self.unconfirmed_boxes.pop(0)

            self._extra_counts[label] += 1
            return True

        return False

    def total_counts(self):
        counts = defaultdict(int)
        for lbl in self.labels.values():
            counts[lbl] += 1
        for lbl, cnt in self._extra_counts.items():
            counts[lbl] += cnt
        return dict(counts)

    def draw(self, frame, w):
        lo = int(self.band_lo)
        hi = int(self.band_hi)
        cv2.rectangle(frame, (0, lo), (w, hi), (0, 255, 255), 1)
        cv2.line(frame, (0, int(self.line_coord)),
                 (w, int(self.line_coord)), (0, 220, 220), 2)
        cv2.putText(frame, "COUNT BAND", (10, lo - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)

# ZONE COUNTER
class ZoneCounter:
    def __init__(self, x1, y1, x2, y2):
        self.x1 = x1; self.y1 = y1; self.x2 = x2; self.y2 = y2

        self.counted_ids: set[int]       = set()
        self.labels:      dict[int, str] = {}

        # ✅ NEW: memory of counted positions
        self.counted_positions: list[tuple] = []
        self.POSITION_TTL = 30   # frames to remember
        self.DIST_THRESH  = 80   # px distance to consider "same object"

    def inside(self, centre):
        cx, cy = centre
        return self.x1 <= cx <= self.x2 and self.y1 <= cy <= self.y2

    def _is_duplicate(self, centre):
        # check if this centre is close to a recently counted one
        for (px, py, ttl) in self.counted_positions:
            if np.hypot(centre[0]-px, centre[1]-py) < self.DIST_THRESH:
                return True
        return False

    def _update_memory(self):
        # decrease TTL and remove expired
        new_mem = []
        for (x, y, ttl) in self.counted_positions:
            if ttl > 1:
                new_mem.append((x, y, ttl-1))
        self.counted_positions = new_mem

    def update(self, tid, centre, label):
        self._update_memory()

        # already counted ID
        if tid in self.counted_ids:
            return False

        # only count if inside zone
        if not self.inside(centre):
            return False

        # ❌ prevent duplicate counting (KEY FIX)
        if self._is_duplicate(centre):
            return False

        # ✅ count
        self.counted_ids.add(tid)
        self.labels[tid] = label

        # store position
        self.counted_positions.append((centre[0], centre[1], self.POSITION_TTL))

        return True

    def draw(self, frame):
        cv2.rectangle(frame, (self.x1, self.y1), (self.x2, self.y2),
                      (0, 255, 255), 2)
        cv2.putText(frame, "COUNT ZONE", (self.x1 + 4, self.y1 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)

    def total_counts(self):
        counts = defaultdict(int)
        for lbl in self.labels.values():
            counts[lbl] += 1
        return dict(counts)

# WEAPON → PERSON ASSIGNER
class WeaponPersonAssigner:
    def __init__(self, max_dist=WEAPON_ASSIGN_DIST):
        self.max_dist     = max_dist
        self.alerted_ids: set[int] = set()
        self.current_armed_ids: set[int] = set()   # ✅ NEW

    def assign(self, weapon_boxes, person_tracks, frame, frame_alerts, global_alerts):
        self.current_armed_ids.clear()  # reset each frame

        # ✅ ALWAYS draw weapon boxes (visible fix)
        for w_box, w_conf in weapon_boxes:
            draw_box(frame, w_box, "weapon", w_conf, (0, 0, 255), thickness=3)

        for w_box, w_conf in weapon_boxes:
            w_centre = box_center(w_box)

            if not person_tracks:
                continue

            nearest_tid, nearest_dist = None, float('inf')

            for tid, p_centre, p_box in person_tracks:
                dist = np.hypot(w_centre[0]-p_centre[0], w_centre[1]-p_centre[1])
                if dist < nearest_dist:
                    nearest_dist = dist
                    nearest_tid  = tid

            if nearest_tid is None or nearest_dist > self.max_dist:
                continue

            # ✅ mark as armed for drawing later
            self.current_armed_ids.add(nearest_tid)

            if nearest_tid not in self.alerted_ids:
                self.alerted_ids.add(nearest_tid)
                msg = "ARMED: person carrying a weapon"
                frame_alerts.append(f"!! {msg}")
                global_alerts.add(msg)

        # ✅ DRAW ARMED PERSON BOXES LAST (prevents overwrite bug)
        for tid, centre, box in person_tracks:
            if tid in self.current_armed_ids:
                x1, y1, x2, y2 = map(int, box)

                cv2.rectangle(frame, (x1-4, y1-4), (x2+4, y2+4), (0,0,255), 3)
                cv2.putText(frame, "ARMED", (x1, y2+20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

class ShelfTheftDetector:

    def __init__(self, x1, y1, x2, y2):
        self.x1, self.y1, self.x2, self.y2 = x1, y1, x2, y2
        self.item_last_seen = {}
        self.item_missing_counter = {}
        self.person_near_item = {}
        self.theft_alerted = set()

    def in_shelf(self, box):
        cx, cy = box_center(box)
        return self.x1 <= cx <= self.x2 and self.y1 <= cy <= self.y2

    def update_items(self, item_detections):
        seen_ids = set()

        for box, conf in item_detections:
            cid = tuple(map(int, box))
            seen_ids.add(cid)
            self.item_last_seen[cid] = box
            self.item_missing_counter[cid] = 0

        for cid in list(self.item_last_seen.keys()):
            if cid not in seen_ids:
                self.item_missing_counter[cid] += 1
                if self.item_missing_counter[cid] > ITEM_DISAPPEAR_FRAMES:
                    del self.item_last_seen[cid]

    def detect_theft(self, frame, persons, items):
        alerts = []

        for p_tid, p_center, p_box in persons:
            for item_box, conf in items:
                item_center = box_center(item_box)
                dist = np.hypot(p_center[0]-item_center[0], p_center[1]-item_center[1])

                if dist < PICKUP_DIST_PX:
                    self.person_near_item[p_tid] = item_box

        for p_tid, item_box in self.person_near_item.items():
            cid = tuple(map(int, item_box))

            if cid not in self.item_last_seen:
                if p_tid not in self.theft_alerted:
                    self.theft_alerted.add(p_tid)
                    alerts.append(f"THEFT ALERT: Person {p_tid} picked item and hid it")

                    cv2.putText(frame, "THEFT", (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 3)

        return alerts

    def draw_shelf(self, frame):
        cv2.rectangle(frame,
                      (self.x1, self.y1),
                      (self.x2, self.y2),
                      (255, 255, 0), 2)

        cv2.putText(frame, "SHELF ZONE",
                    (self.x1+5, self.y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (255,255,0), 2)

# SURVEILLANCE MODE
def process_security(video_path: str):
    if video_path is None:
        return None, "No video provided."

    cap  = cv2.VideoCapture(video_path)
    w, h = int(cap.get(3)), int(cap.get(4))
    fps  = max(1, int(cap.get(5)))

    zx1  = int(w * 0.25)
    zx2  = int(w * 0.75)
    zone = ZoneCounter(x1=zx1, y1=0, x2=zx2, y2=h)

    assigner = WeaponPersonAssigner(max_dist=250)

    # ✅ NEW: Shelf detector
    shelf = ShelfTheftDetector(
        int(w * SHELF_X1_FRAC),
        int(h * SHELF_Y1_FRAC),
        int(w * SHELF_X2_FRAC),
        int(h * SHELF_Y2_FRAC)
    )

    global_alerts = set()

    out_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    out      = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_alerts = []

        zone.draw(frame)
        shelf.draw_shelf(frame)  # ✅ NEW

        results = GENERAL_MODEL.track(
            frame, persist=True, tracker="bytetrack.yaml",
            conf=GENERAL_CONF, classes=SECURITY_CLASSES, verbose=False,
        )[0]

        weapon_results = WEAPON_MODEL.predict(
            frame, conf=0.25, imgsz=960, verbose=False
        )[0]

        bag_centres   = []
        person_tracks = []

        # ✅ NEW
        persons = []
        items   = []

        for b in results.boxes:
            cls    = int(b.cls[0])
            conf   = float(b.conf[0])
            box    = b.xyxy[0].cpu().numpy()
            label  = get_label(cls)
            color  = COLOR_MAP.get(label, (200, 200, 200))
            centre = box_center(box)

            if b.id is not None:
                tid = int(b.id)
                zone.update(tid, centre, label)
                draw_box(frame, box, label, conf, color)

                if label == "bag":
                    bag_centres.append((*centre, tid))

                if label == "person":
                    person_tracks.append((tid, centre, box))
                    persons.append((tid, centre, box))  # ✅ NEW
            else:
                draw_box(frame, box, label, conf, color)

            # ✅ ITEM LOGIC
            if label in ["bag", "bottle", "cup", "book"]:
                if shelf.in_shelf(box):
                    items.append((box, conf))
                    draw_box(frame, box, "ITEM", conf, (255, 255, 0))

        # ✅ UPDATE ITEM TRACKING
        shelf.update_items(items)

        # ❌ REMOVED LOITERING BLOCK COMPLETELY

        # ---------------- WEAPON ----------------
        if len(weapon_results.boxes):
            fused = fuse_detections([
                (b.xyxy[0].cpu().numpy(), float(b.conf[0]), 999)
                for b in weapon_results.boxes if float(b.conf[0]) > 0.30
            ])

            clean_weapons = []
            for box, conf, _ in fused:
                if conf > 0.35:
                    clean_weapons.append((box, conf))

            assigner.assign(
                clean_weapons,
                person_tracks,
                frame,
                frame_alerts,
                global_alerts
            )

        # ✅ NEW THEFT DETECTION
        theft_alerts = shelf.detect_theft(frame, persons, items)
        for a in theft_alerts:
            frame_alerts.append("!! " + a)
            global_alerts.add(a)

        draw_counts(frame, zone.total_counts(), w, h)
        overlay_alerts(frame, frame_alerts)
        out.write(frame)

    cap.release()
    out.release()

    counts = zone.total_counts()
    lines  = ["─"*34, "UNIQUE OBJECT COUNTS (zone entry)"]
    lines += [f"  {k}: {v}" for k, v in sorted(counts.items())]

    if global_alerts:
        lines += ["", "ALERTS"]
        lines += [f"  !! {a}" for a in sorted(global_alerts)]
    else:
        lines.append("\nNo alerts triggered.")

    return out_path, "\n".join(lines)

# TRAFFIC MODE
def process_traffic(video_path: str):
    if video_path is None:
        return None, "No video provided."

    cap  = cv2.VideoCapture(video_path)
    w, h = int(cap.get(3)), int(cap.get(4))
    fps  = max(1, int(cap.get(5)))

    line_y  = int(h * LINE_FRAC)
    counter = RobustLineCrossingCounter(line_coord=line_y, axis="y")

    out_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    out      = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    prev_centres:  dict[int, tuple] = {}
    speed_history: dict[int, list]  = defaultdict(list)
    speed_now:     dict[int, float] = defaultdict(float)
    speed_alerted: set[int]         = set()
    speed_alerts:  set[str]         = set()
    prev_frame_tids: set[int]       = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_alerts: list[str] = []
        counter.tick_ghosts()
        counter.draw(frame, w)

        results = GENERAL_MODEL.track(
            frame, persist=True, tracker="bytetrack.yaml",
            conf=GENERAL_CONF, classes=TRAFFIC_CLASSES, verbose=False,
        )[0]

        current_frame_tids: set[int] = set()

        for b in results.boxes:
            cls    = int(b.cls[0])
            conf   = float(b.conf[0])
            box    = b.xyxy[0].cpu().numpy()
            label  = get_label(cls)
            colour = COLOR_MAP.get(label, (200, 200, 200))
            centre = box_center(box)
            cx, cy = centre

            if b.id is None:
                box_key = (round(cx / 20) * 20, round(cy / 20) * 20)
                counted = counter.update_unconfirmed(centre, label, box_key)
                col = (0, 80, 255) if counted else colour
                draw_box(frame, box, label, conf, col)
                continue

            tid = int(b.id)
            current_frame_tids.add(tid)

            # SPEED CALIBRATION
            REAL_WORLD_SCALE = 0.05
            MIN_MOVEMENT_PX = 2
            MAX_REASONABLE_KMH = 180

            if tid in prev_centres:
                px, py = prev_centres[tid]
                dist_px = np.hypot(cx - px, cy - py)

                if dist_px < MIN_MOVEMENT_PX:
                    kmh = 0.0
                else:
                    meters = dist_px * REAL_WORLD_SCALE
                    kmh = meters * fps * 3.6
                    if kmh > MAX_REASONABLE_KMH:
                        kmh = 0.0

                speed_history[tid].append(kmh)
                if len(speed_history[tid]) > SPEED_SMOOTH_N:
                    speed_history[tid].pop(0)

                smooth_kmh = float(np.mean(speed_history[tid]))
                speed_now[tid] = smooth_kmh

                if smooth_kmh > SPEED_LIMIT_KMH and tid not in speed_alerted:
                    speed_alerted.add(tid)
                    msg = f"SPEED ALERT: {label} {conf:.0%} @ {smooth_kmh:.1f} km/h"
                    frame_alerts.append(f"!! {msg}")
                    speed_alerts.add(msg)

            prev_centres[tid] = centre
            counter.update(tid, centre, label)

            spd = speed_now.get(tid, 0.0)
            color_draw = (0, 0, 255) if spd > SPEED_LIMIT_KMH else colour
            draw_box_with_speed(frame, box, label, conf, spd, color_draw)

            if spd > SPEED_LIMIT_KMH:
                x1, y1, x2, y2 = map(int, box)
                overlay = frame.copy()
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 200), -1)
                cv2.addWeighted(overlay, 0.25, frame, 0.75, 0, frame)

        for vanished_tid in prev_frame_tids - current_frame_tids:
            if vanished_tid in prev_centres:
                counter.expire_track(vanished_tid, prev_centres[vanished_tid])

        prev_frame_tids = current_frame_tids
        draw_counts(frame, counter.total_counts(), w, h)
        overlay_alerts(
            frame, frame_alerts,
            footer=(f"Limit: {SPEED_LIMIT_KMH} km/h  |  "
                    f"Base: {BASE_PX_PER_METER} px/m  |  "
                    f"Band: ±{LINE_BAND_HALF}px  |  Perspective: ON")
        )
        out.write(frame)

    cap.release()
    out.release()

    counts = counter.total_counts()
    lines  = ["─"*34, "UNIQUE VEHICLE COUNTS (robust band-crossing)"]
    lines += [f"  {k}: {v}" for k, v in sorted(counts.items())]
    if speed_alerts:
        lines += ["", "SPEED ALERTS (one per vehicle)"]
        lines += [f"  !! {a}" for a in sorted(speed_alerts)]
    else:
        lines.append(f"\nNo vehicles exceeded {SPEED_LIMIT_KMH} km/h.")
    return out_path, "\n".join(lines)


# DISPATCHER
def run_analysis(video, mode):
    if mode == "🚦 Traffic":
        return process_traffic(video)
    return process_security(video)


# GRADIO UI
with gr.Blocks(title="AI Sentinel") as demo:
    gr.Markdown("# Traffic Monitoring and Surveillance System")
    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label="Input Video")
            mode_sel = gr.Radio(
                choices=["🚦 Traffic", "🔍 Surveillance"],
                value="🚦 Traffic",
                label="Analysis Mode",
            )
            gr.Markdown(
                "**Traffic**  "
                "one alert per vehicle >90 km/h.  \n"
                "**Surveillance** — weapon alert,loitering alert."
            )
            btn = gr.Button("▶ Run Analysis", variant="primary")
        with gr.Column(scale=1):
            video_out = gr.Video(label="Annotated Output")
            report    = gr.Textbox(label="Report", lines=20, interactive=False)

    btn.click(run_analysis, inputs=[video_in, mode_sel], outputs=[video_out, report])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://49be445e269506ad0a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 100.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [15]:
import cv2
import numpy as np
import tempfile
import gradio as gr
import os
import torch
from collections import defaultdict
from ultralytics import YOLO, RTDETR
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from huggingface_hub import hf_hub_download

# ================== WEAPON MODEL SETUP ==================
MODEL_REPO = "Subh775/Threat-Detection-YOLOv8n"
BATCH_SIZE = 16

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

try:
    model_path = hf_hub_download(repo_id=MODEL_REPO, filename="weights/best.pt")
    WEAPON_MODEL = YOLO(model_path)
    WEAPON_CONF = 0.40
except Exception as e:
    print(f"Error loading Weapon model: {e}")

# ================== SHOPLIFTING CONSTANTS ==================
SHELF_X1_FRAC = 0.10
SHELF_X2_FRAC = 0.90
SHELF_Y1_FRAC = 0.20
SHELF_Y2_FRAC = 0.80

ITEM_DISAPPEAR_FRAMES   = 15
PICKUP_DIST_PX          = 120
SPEED_LIMIT_KMH         = 90
SPEED_SMOOTH_N          = 7
LINE_FRAC               = 0.50
LINE_BAND_HALF          = 30
GHOST_FRAMES            = 8
UNCONFIRMED_BAND_MARGIN = LINE_BAND_HALF + 20
BASE_PX_PER_METER       = 8.0
ZONE_X_START_FRAC       = 0.40
ZONE_X_END_FRAC         = 0.60
LOITER_FRAMES           = 90
LOITER_DIST_PX          = 120
WEAPON_ASSIGN_DIST      = 180

PROJECT_PATH = "/content/drive/MyDrive/WeaponDetectionProject"
BEST_WEIGHTS = os.path.join(PROJECT_PATH, "weapon_model", "weights", "best.pt")

try:
    GENERAL_MODEL
except NameError:
    print("Loading general model…")
    GENERAL_MODEL = RTDETR("rtdetr-l.pt")

GENERAL_CONF = globals().get("GENERAL_CONF", 0.60)
WEAPON_CONF  = globals().get("WEAPON_CONF",  0.40)
SECURITY_CLASSES = [0, 1, 2, 3, 5, 7, 24, 26, 28]
TRAFFIC_CLASSES  = [1, 2, 3, 5, 7]
COCO_NAMES       = GENERAL_MODEL.names

COLOR_MAP = {
    'person':    (0,   255,   0),
    'car':       (255, 200,   0),
    'motorbike': (0,   200, 255),
    'bus':       (255, 100,   0),
    'truck':     (200,   0, 255),
    'bicycle':   (0,   255, 200),
    'bag':       (255, 255,   0),
    'weapon':    (0,     0, 255),
}

# ================== HELPERS ==================
def get_label(cls):
    name = COCO_NAMES.get(cls, "object").lower()
    if cls in [24, 26, 28]:
        return "bag"
    return name


def box_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def perspective_px_per_meter(cy: float, frame_height: int) -> float:
    scale = 1.0 + (cy / frame_height)
    return BASE_PX_PER_METER / scale


def iou(b1, b2):
    xi1, yi1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    xi2, yi2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-6)


def fuse_detections(dets, iou_thresh=0.45):
    filtered = []
    for det in sorted(dets, key=lambda x: x[1], reverse=True):
        if all(iou(det[0], fd[0]) < iou_thresh for fd in filtered):
            filtered.append(det)
    return filtered


def draw_box(frame, box, label, conf, color, thickness=2):
    x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    tag = f"{label} {conf:.0%}"
    (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    label_y1 = max(y1 - th - 6, 0)
    label_y2 = max(y1, th + 6)
    cv2.rectangle(frame, (x1, label_y1), (x1 + tw + 4, label_y2), color, -1)
    cv2.putText(frame, tag, (x1 + 2, label_y2 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)


def draw_box_with_speed(frame, box, label, conf, speed_kmh, color):
    x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    tag = (f"{label} {conf:.0%}  {speed_kmh:.1f}km/h"
           if speed_kmh > 0 else f"{label} {conf:.0%}")
    (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 2)
    label_y1 = max(y1 - th - 6, 0)
    label_y2 = max(y1, th + 6)
    cv2.rectangle(frame, (x1, label_y1), (x1 + tw + 4, label_y2), color, -1)
    cv2.putText(frame, tag, (x1 + 2, label_y2 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 0, 0), 2)


def overlay_alerts(frame, alert_lines, footer=None):
    y = 30
    for line in alert_lines:
        cv2.putText(frame, line, (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)
        y += 28
    if footer:
        cv2.putText(frame, footer, (10, frame.shape[0] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)


def draw_counts(frame, counts, w, h):
    y_off = h - 20 - len(counts) * 22
    for lbl, cnt in sorted(counts.items()):
        cv2.putText(frame, f"{lbl}: {cnt}", (w - 170, y_off),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    COLOR_MAP.get(lbl, (255, 255, 255)), 2)
        y_off += 22


# ================== ROBUST LINE-CROSSING COUNTER ==================
class RobustLineCrossingCounter:
    def __init__(self, line_coord: float, axis: str = "y"):
        self.line_coord = line_coord
        self.axis       = axis
        self.band_lo    = line_coord - LINE_BAND_HALF
        self.band_hi    = line_coord + LINE_BAND_HALF

        self.prev_side   = {}
        self.counted_ids = set()
        self.labels      = {}

        self.ghosts = {}
        self.unconfirmed_boxes = []

        self._extra_counts = defaultdict(int)

    def _coord(self, centre):
        return centre[1] if self.axis == "y" else centre[0]

    def _side(self, centre):
        return 1 if self._coord(centre) >= self.line_coord else -1

    def _in_band(self, centre):
        c = self._coord(centre)
        return self.band_lo <= c <= self.band_hi

    def tick_ghosts(self):
        expired = [tid for tid, (_, _, ttl) in self.ghosts.items() if ttl <= 1]
        for tid in expired:
            del self.ghosts[tid]
        for tid in list(self.ghosts):
            side, c, ttl = self.ghosts[tid]
            self.ghosts[tid] = (side, c, ttl - 1)

    def update(self, tid, centre, label):
        side = self._side(centre)

        if tid in self.ghosts:
            self.prev_side[tid] = self.ghosts[tid][0]
            del self.ghosts[tid]

        if tid in self.counted_ids:
            self.prev_side[tid] = side
            return False

        if tid not in self.prev_side:
            for gid, (gside, gc, _) in self.ghosts.items():
                if np.hypot(centre[0]-gc[0], centre[1]-gc[1]) < 80:
                    self.prev_side[tid] = gside
                    del self.ghosts[gid]
                    break
            if tid not in self.prev_side:
                self.prev_side[tid] = side
                return False

        prev    = self.prev_side[tid]
        crossed = (prev != side)

        if crossed:
            self.counted_ids.add(tid)
            self.labels[tid] = label
            self.prev_side[tid] = side
            return True

        self.prev_side[tid] = side
        return False

    def expire_track(self, tid, centre):
        if tid not in self.counted_ids and tid in self.prev_side:
            self.ghosts[tid] = (self.prev_side[tid], centre, GHOST_FRAMES)

    def update_unconfirmed(self, centre, label, box):
        for prev_box in self.unconfirmed_boxes:
            if iou(prev_box, box) > 0.5:
                return False

        if self._in_band(centre):
            self.unconfirmed_boxes.append(box)
            if len(self.unconfirmed_boxes) > 50:
                self.unconfirmed_boxes.pop(0)
            self._extra_counts[label] += 1
            return True

        return False

    def total_counts(self):
        counts = defaultdict(int)
        for lbl in self.labels.values():
            counts[lbl] += 1
        for lbl, cnt in self._extra_counts.items():
            counts[lbl] += cnt
        return dict(counts)

    def draw(self, frame, w):
        lo = int(self.band_lo)
        hi = int(self.band_hi)
        cv2.rectangle(frame, (0, lo), (w, hi), (0, 255, 255), 1)
        cv2.line(frame, (0, int(self.line_coord)),
                 (w, int(self.line_coord)), (0, 220, 220), 2)
        cv2.putText(frame, "COUNT BAND", (10, lo - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)


# ================== ZONE COUNTER (kept, unused in security now) ==================
class ZoneCounter:
    def __init__(self, x1, y1, x2, y2):
        self.x1 = x1; self.y1 = y1; self.x2 = x2; self.y2 = y2
        self.counted_ids: set         = set()
        self.labels:      dict        = {}
        self.counted_positions: list  = []
        self.POSITION_TTL = 80
        self.DIST_THRESH  = 120
        self.prev_inside: dict        = {}

    def inside(self, centre):
        cx, cy = centre
        return self.x1 <= cx <= self.x2 and self.y1 <= cy <= self.y2

    def _is_duplicate(self, centre):
        for (px, py, ttl) in self.counted_positions:
            if np.hypot(centre[0]-px, centre[1]-py) < self.DIST_THRESH:
                return True
        return False

    def _update_memory(self):
        self.counted_positions = [
            (x, y, ttl-1) for (x, y, ttl) in self.counted_positions if ttl > 1
        ]

    def update(self, tid, centre, label):
        self._update_memory()
        inside_now  = self.inside(centre)
        inside_prev = self.prev_inside.get(tid, False)
        self.prev_inside[tid] = inside_now

        if tid in self.counted_ids:
            return False
        if not (inside_now and not inside_prev):
            return False
        if self._is_duplicate(centre):
            return False

        self.counted_ids.add(tid)
        self.labels[tid] = label
        self.counted_positions.append((centre[0], centre[1], self.POSITION_TTL))
        return True

    def draw(self, frame):
        cv2.rectangle(frame, (self.x1, self.y1), (self.x2, self.y2),
                      (0, 255, 255), 2)
        cv2.putText(frame, "COUNT ZONE", (self.x1 + 4, self.y1 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)

    def total_counts(self):
        counts = defaultdict(int)
        for lbl in self.labels.values():
            counts[lbl] += 1
        return dict(counts)


# ================== WEAPON → PERSON ASSIGNER ==================
class WeaponPersonAssigner:
    def __init__(self, max_dist=WEAPON_ASSIGN_DIST):
        self.max_dist          = max_dist
        self.alerted_ids:      set = set()
        self.current_armed_ids: set = set()

    def assign(self, weapon_boxes, person_tracks, frame, frame_alerts, global_alerts):
        self.current_armed_ids.clear()

        for w_box, w_conf in weapon_boxes:
            draw_box(frame, w_box, "weapon", w_conf, (0, 0, 255), thickness=3)

        for w_box, w_conf in weapon_boxes:
            w_centre = box_center(w_box)

            if not person_tracks:
                continue

            nearest_tid, nearest_dist = None, float('inf')

            for tid, p_centre, p_box in person_tracks:
                dist = np.hypot(w_centre[0]-p_centre[0], w_centre[1]-p_centre[1])
                if dist < nearest_dist:
                    nearest_dist = dist
                    nearest_tid  = tid

            if nearest_tid is None or nearest_dist > self.max_dist:
                continue

            self.current_armed_ids.add(nearest_tid)

            if nearest_tid not in self.alerted_ids:
                self.alerted_ids.add(nearest_tid)
                msg = "ARMED: person carrying a weapon"
                frame_alerts.append(f"!! {msg}")
                global_alerts.add(msg)

        for tid, centre, box in person_tracks:
            if tid in self.current_armed_ids:
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1-4, y1-4), (x2+4, y2+4), (0, 0, 255), 3)
                cv2.putText(frame, "ARMED", (x1, y2+20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)


# ================== SHELF THEFT DETECTOR ==================
class ShelfTheftDetector:
    def __init__(self, x1, y1, x2, y2):
        self.x1, self.y1, self.x2, self.y2 = x1, y1, x2, y2
        self.item_last_seen      = {}
        self.item_missing_counter = {}
        self.person_near_item    = {}
        self.theft_alerted       = set()

    def in_shelf(self, box):
        cx, cy = box_center(box)
        return self.x1 <= cx <= self.x2 and self.y1 <= cy <= self.y2

    def update_items(self, item_detections):
        seen_ids = set()
        for box, conf in item_detections:
            cid = tuple(map(int, box))
            seen_ids.add(cid)
            self.item_last_seen[cid]       = box
            self.item_missing_counter[cid] = 0

        for cid in list(self.item_last_seen.keys()):
            if cid not in seen_ids:
                self.item_missing_counter[cid] += 1
                if self.item_missing_counter[cid] > ITEM_DISAPPEAR_FRAMES:
                    del self.item_last_seen[cid]

    def detect_theft(self, frame, persons, items):
        alerts = []

        for p_tid, p_center, p_box in persons:
            for item_box, conf in items:
                item_center = box_center(item_box)
                dist = np.hypot(p_center[0]-item_center[0], p_center[1]-item_center[1])
                if dist < PICKUP_DIST_PX:
                    self.person_near_item[p_tid] = item_box

        for p_tid, item_box in self.person_near_item.items():
            cid = tuple(map(int, item_box))
            if cid not in self.item_last_seen:
                if p_tid not in self.theft_alerted:
                    self.theft_alerted.add(p_tid)
                    alerts.append(f"THEFT ALERT: Person {p_tid} picked item and hid it")
                    cv2.putText(frame, "THEFT", (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

        return alerts

    def draw_shelf(self, frame):
        cv2.rectangle(frame, (self.x1, self.y1), (self.x2, self.y2),
                      (255, 255, 0), 2)
        cv2.putText(frame, "SHELF ZONE", (self.x1+5, self.y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)


# ================== SECURITY MODE ==================
def process_security(video_path: str):
    if video_path is None:
        return None, "No video provided."

    cap = cv2.VideoCapture(video_path)
    w, h = int(cap.get(3)), int(cap.get(4))
    fps  = max(1, int(cap.get(5)))

    # --- Line-crossing counter replaces ZoneCounter ---
    line_y = int(h * 0.50)   # mid-frame horizontal line; adjust as needed
    zone   = RobustLineCrossingCounter(line_coord=line_y, axis="y")

    assigner = WeaponPersonAssigner(max_dist=250)

    shelf = ShelfTheftDetector(
        int(w * SHELF_X1_FRAC),
        int(h * SHELF_Y1_FRAC),
        int(w * SHELF_X2_FRAC),
        int(h * SHELF_Y2_FRAC),
    )

    global_alerts:   set  = set()
    prev_centres:    dict = {}
    prev_frame_tids: set  = set()

    out_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_alerts:       list = []
        current_frame_tids: set  = set()

        # --- draw overlays ---
        zone.draw(frame, w)          # FIX: was zone.draw(frame) — needs w
        zone.tick_ghosts()           # FIX: was counter.tick_ghosts() — wrong name
        shelf.draw_shelf(frame)

        results = GENERAL_MODEL.track(
            frame, persist=True, conf=GENERAL_CONF,
            classes=SECURITY_CLASSES, verbose=False,
        )[0]

        person_tracks: list = []
        persons:       list = []
        items:         list = []

        # --- detection loop ---
        if results.boxes is not None:
            for b in results.boxes:
                cls   = int(b.cls[0])
                conf  = float(b.conf[0])
                box   = b.xyxy[0].cpu().numpy()
                label = get_label(cls)
                tid   = int(b.id) if b.id is not None else -1

                # tracked detections
                if tid != -1:                                    # FIX: single block, no duplicate
                    zone.update(tid, box_center(box), label)
                    prev_centres[tid] = box_center(box)
                    current_frame_tids.add(tid)

                    if label == "person":
                        person_tracks.append((tid, box_center(box), box))   # 3-tuple
                        persons.append((tid, box_center(box), box))

                # item logic (runs for all detections, tracked or not)
                if label in ["bag", "bottle", "cup", "book"]:
                    if shelf.in_shelf(box):
                        items.append((box, conf))
                        draw_box(frame, box, "ITEM", conf, (255, 255, 0))

                color = COLOR_MAP.get(label, (255, 255, 255))
                draw_box(frame, box, label, conf, color)

        # --- item tracking ---
        shelf.update_items(items)

        # --- weapon detection (every 5 frames) ---
        weapon_boxes = []
        if frame_count % 5 == 0:
            w_results = WEAPON_MODEL.predict(frame, conf=WEAPON_CONF, verbose=False)[0]
            if w_results.boxes is not None:
                for wb in w_results.boxes:
                    w_box   = wb.xyxy[0].cpu().numpy()
                    w_score = float(wb.conf[0])
                    weapon_boxes.append((w_box.tolist(), w_score))

        # --- weapon → person assignment ---
        if weapon_boxes:
            assigner.assign(
                weapon_boxes,
                person_tracks,      # FIX: already 3-tuples, no extra unpack needed
                frame,
                frame_alerts,
                global_alerts,
            )

        # --- theft detection ---
        theft_alerts = shelf.detect_theft(frame, persons, items)
        for a in theft_alerts:
            frame_alerts.append("!! " + a)
            global_alerts.add(a)

        # --- expire vanished tracks (ghost suppression) ---
        for vanished_tid in prev_frame_tids - current_frame_tids:
            if vanished_tid in prev_centres:
                zone.expire_track(vanished_tid, prev_centres[vanished_tid])
        prev_frame_tids = current_frame_tids

        draw_counts(frame, zone.total_counts(), w, h)
        overlay_alerts(frame, frame_alerts)

        out.write(frame)
        frame_count += 1

    cap.release()
    out.release()

    final_counts = zone.total_counts()
    report_text  = "--- SURVEILLANCE REPORT ---\n"
    report_text += "\n".join([f"{k.upper()}: {v}" for k, v in final_counts.items()])
    report_text += "\n\nACTIVE ALERTS:\n"
    report_text += (
        "\n".join([f"- {a}" for a in global_alerts])
        if global_alerts else "No alerts."
    )
    return out_path, report_text


# ================== TRAFFIC MODE (UNCHANGED) ==================
def process_traffic(video_path: str):
    if video_path is None:
        return None, "No video provided."

    cap  = cv2.VideoCapture(video_path)
    w, h = int(cap.get(3)), int(cap.get(4))
    fps  = max(1, int(cap.get(5)))

    line_y  = int(h * LINE_FRAC)
    counter = RobustLineCrossingCounter(line_coord=line_y, axis="y")

    out_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    out      = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    prev_centres:  dict = {}
    speed_history: dict = defaultdict(list)
    speed_now:     dict = defaultdict(float)
    speed_alerted: set  = set()
    speed_alerts:  set  = set()
    prev_frame_tids: set = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_alerts: list = []
        counter.tick_ghosts()
        counter.draw(frame, w)

        results = GENERAL_MODEL.track(
            frame, persist=True, tracker="bytetrack.yaml",
            conf=GENERAL_CONF, classes=TRAFFIC_CLASSES, verbose=False,
        )[0]

        current_frame_tids: set = set()

        for b in results.boxes:
            cls    = int(b.cls[0])
            conf   = float(b.conf[0])
            box    = b.xyxy[0].cpu().numpy()
            label  = get_label(cls)
            colour = COLOR_MAP.get(label, (200, 200, 200))
            centre = box_center(box)
            cx, cy = centre

            if b.id is None:
                box_key = (round(cx / 20) * 20, round(cy / 20) * 20)
                counted = counter.update_unconfirmed(centre, label, box_key)
                col = (0, 80, 255) if counted else colour
                draw_box(frame, box, label, conf, col)
                continue

            tid = int(b.id)
            current_frame_tids.add(tid)

            REAL_WORLD_SCALE    = 0.05
            MIN_MOVEMENT_PX     = 2
            MAX_REASONABLE_KMH  = 180

            if tid in prev_centres:
                px, py  = prev_centres[tid]
                dist_px = np.hypot(cx - px, cy - py)

                if dist_px < MIN_MOVEMENT_PX:
                    kmh = 0.0
                else:
                    meters = dist_px * REAL_WORLD_SCALE
                    kmh    = meters * fps * 3.6
                    if kmh > MAX_REASONABLE_KMH:
                        kmh = 0.0

                speed_history[tid].append(kmh)
                if len(speed_history[tid]) > SPEED_SMOOTH_N:
                    speed_history[tid].pop(0)

                smooth_kmh      = float(np.mean(speed_history[tid]))
                speed_now[tid]  = smooth_kmh

                if smooth_kmh > SPEED_LIMIT_KMH and tid not in speed_alerted:
                    speed_alerted.add(tid)
                    msg = f"SPEED ALERT: {label} {conf:.0%} @ {smooth_kmh:.1f} km/h"
                    frame_alerts.append(f"!! {msg}")
                    speed_alerts.add(msg)

            prev_centres[tid] = centre
            counter.update(tid, centre, label)

            spd        = speed_now.get(tid, 0.0)
            color_draw = (0, 0, 255) if spd > SPEED_LIMIT_KMH else colour
            draw_box_with_speed(frame, box, label, conf, spd, color_draw)

            if spd > SPEED_LIMIT_KMH:
                x1, y1, x2, y2 = map(int, box)
                overlay = frame.copy()
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 200), -1)
                cv2.addWeighted(overlay, 0.25, frame, 0.75, 0, frame)

        for vanished_tid in prev_frame_tids - current_frame_tids:
            if vanished_tid in prev_centres:
                counter.expire_track(vanished_tid, prev_centres[vanished_tid])

        prev_frame_tids = current_frame_tids
        draw_counts(frame, counter.total_counts(), w, h)
        overlay_alerts(
            frame, frame_alerts,
            footer=(f"Limit: {SPEED_LIMIT_KMH} km/h  |  "
                    f"Base: {BASE_PX_PER_METER} px/m  |  "
                    f"Band: ±{LINE_BAND_HALF}px  |  Perspective: ON")
        )
        out.write(frame)

    cap.release()
    out.release()

    counts = counter.total_counts()
    lines  = ["─"*34, "UNIQUE VEHICLE COUNTS (robust band-crossing)"]
    lines += [f"  {k}: {v}" for k, v in sorted(counts.items())]
    if speed_alerts:
        lines += ["", "SPEED ALERTS (one per vehicle)"]
        lines += [f"  !! {a}" for a in sorted(speed_alerts)]
    else:
        lines.append(f"\nNo vehicles exceeded {SPEED_LIMIT_KMH} km/h.")
    return out_path, "\n".join(lines)


# ================== DISPATCHER ==================
def run_analysis(video, mode):
    if mode == "🚦 Traffic":
        return process_traffic(video)
    return process_security(video)


# ================== GRADIO UI ==================
with gr.Blocks(title="AI Sentinel") as demo:
    gr.Markdown("# Traffic Monitoring and Surveillance System")
    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label="Input Video")
            mode_sel = gr.Radio(
                choices=["🚦 Traffic", "🔍 Surveillance"],
                value="🚦 Traffic",
                label="Analysis Mode",
            )
            gr.Markdown(
                "**Traffic** — one alert per vehicle >90 km/h.  \n"
                "**Surveillance** — weapon alert, theft alert."
            )
            btn = gr.Button("▶ Run Analysis", variant="primary")
        with gr.Column(scale=1):
            video_out = gr.Video(label="Annotated Output")
            report    = gr.Textbox(label="Report", lines=20, interactive=False)

    btn.click(run_analysis, inputs=[video_in, mode_sel], outputs=[video_out, report])

demo.launch(share=True)

Using device: GPU
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23d361eaf654bc57df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
